# Joint System Composition Tutorial

This tutorial explores how to compose and run joint 6-D architecture systems in Computronium.

In [ ]:
# Install if needed
!pip install -e . --quiet

## Understanding Coordinates

A 6-D coordinate specifies a complete architecture:

`substrate/geometry/dynamics/plasticity/credit/update`

Let's explore the available options for each axis:

In [ ]:
from computronium.ontology import (
    CREDIT_REGISTRY,
    DYNAMICS_REGISTRY,
    GEOMETRY_REGISTRY,
    PLASTICITY_REGISTRY,
    SUBSTRATE_REGISTRY,
    UPDATE_REGISTRY,
)

print("=== SUBSTRATES ===")
for k in sorted(SUBSTRATE_REGISTRY.keys()):
    print(f"  {k}")

print("\n=== GEOMETRIES ===")
for k in sorted(GEOMETRY_REGISTRY.keys()):
    print(f"  {k}")

print("\n=== DYNAMICS ===")
for k in sorted(DYNAMICS_REGISTRY.keys()):
    print(f"  {k}")

print("\n=== PLASTICITIES ===")
for k in sorted(PLASTICITY_REGISTRY.keys()):
    print(f"  {k}")

print("\n=== CREDITS ===")
for k in sorted(CREDIT_REGISTRY.keys()):
    print(f"  {k}")

print("\n=== UPDATES ===")
for k in sorted(UPDATE_REGISTRY.keys()):
    print(f"  {k}")

## Composing a Joint System

Use `compose_joint_system` to create a system from a coordinate string:

In [ ]:
from computronium.core.joint import compose_joint_system
from computronium.core.joint.transition import PlasticityConfig

# Example 1: Standard backprop MLP
coord1 = "digital/feedforward/instantaneous/null/gradient/euclidean"
system1 = compose_joint_system(coord1, input_dim=784, hidden_dim=256, output_dim=10)
print(f"System 1: {system1}")
print(f"  Parameters: {sum(p.numel() for p in system1.parameters()):,}")

# Example 2: Equilibrium Propagation with Routing Plasticity
coord2 = "digital/recurrent/energy_minimization/routing/equilibrium_prop/adam"
system2 = compose_joint_system(
    coord2,
    input_dim=784,
    hidden_dim=256,
    output_dim=10,
    plasticity_config=PlasticityConfig(
        plasticity_type="routing",
        plastic_state_dims={"gate_logits": 256, "active_routes": 256},
    ),
)
print(f"\nSystem 2: {system2}")
print(f"  Parameters: {sum(p.numel() for p in system2.parameters()):,}")

# Example 3: Memristive substrate with Predictive Settling
coord3 = "memristive/recurrent/predictive_settling/substrate_coupled/pepita/lion"
system3 = compose_joint_system(
    coord3,
    input_dim=784,
    hidden_dim=256,
    output_dim=10,
    plasticity_config=PlasticityConfig(
        plasticity_type="substrate_coupled", plastic_state_dims={}
    ),
)
print(f"\nSystem 3: {system3}")
print(f"  Parameters: {sum(p.numel() for p in system3.parameters()):,}")

## Running a System

Let's train and evaluate a composed system:

In [ ]:
import torch

from computronium.evaluation.benchmarks import mnist_benchmark

device = "cuda" if torch.cuda.is_available() else "cpu"
system1 = system1.to(device)

# Quick training
optimizer = torch.optim.Adam(system1.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss()

from computronium.domains import VisionTask

task = VisionTask(name="mnist", dataset_name="mnist", batch_size=128)
task.setup()

system1.train()
for epoch in range(3):
    for x, y in task.train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = system1(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch}: loss = {loss.item():.4f}")

In [ ]:
# Evaluate
result = mnist_benchmark(system1)
print(f"Validation Accuracy: {result.metrics.get('validation_accuracy', 'N/A'):.4f}")

## Invalid Combinations

Not all combinations are valid. The system will validate and reject invalid coordinates:

In [ ]:
# This will fail - instantaneous dynamics with recurrent geometry and gradient credit
try:
    bad_coord = "digital/recurrent/instantaneous/null/gradient/euclidean"
    bad_system = compose_joint_system(
        bad_coord, input_dim=784, hidden_dim=256, output_dim=10
    )
except ValueError as e:
    print(f"Expected error: {e}")

# Check valid combinations programmatically
from computronium.core.joint.transition import _is_valid_combination

print("\nValid combinations:")
print(
    f"  digital/recurrent/energy_minimization/null/gradient/euclidean: {_is_valid_combination('digital', 'recurrent', 'energy_minimization', 'null', 'gradient', 'euclidean')}"
)
print(
    f"  digital/recurrent/instantaneous/null/gradient/euclidean: {_is_valid_combination('digital', 'recurrent', 'instantaneous', 'null', 'gradient', 'euclidean')}"
)
print(
    f"  digital/feedforward/instantaneous/routing/gradient/euclidean: {_is_valid_combination('digital', 'feedforward', 'instantaneous', 'routing', 'gradient', 'euclidean')}"
)

## Next Steps

- Run [campaigns](campaigns.ipynb) for systematic comparisons
- Learn about [stability analysis](stability_analysis.ipynb)
- Explore [benchmark contributions](benchmark_contributions.ipynb)